In [ ]:
"""
ME 144/244: Modeling, Simulation, and Digital Twins of Drone-Based Systems 
Project 5: Optimization of Advanced Materials for Drones
Fall 2026 Semester

STUDENT VERSION
"""
import numpy as np
import matplotlib.pyplot as plt
import pickle

plt.rcParams['animation.embed_limit'] = 2**128


# Step 0

In [ ]:
## Given Genetic Algorithm Parameters
P = 10  # number of design strings to breed
K = 10  # number of offspring design strings 
G = 5000  # maximum number of generations
S = 200  # total number of design strings per generation
dv = 5 # number of design variables per string

## Given Properties for Material 1
k1 = 90*10**9         # Phase 1 bulk modulus, [Pa]
mu1 = 20*10**9        # Phase 1 shear modulus, [Pa]
sig1E = 1.1*10**7       # Phase 1 electrical conductivity, [S/m]
K1 = 4.5          # Phase 1 thermal conductivity, [W/m/K]
k_effD = 125*10**9    # Desired effective bulk modulus, [Pa]
mu_effD = 50*10**9    # Desired effective shear modulus, [Pa]
sigE_effD = 2.2*10**7   # Desired effective electrical conductivity,[S/m]
K_effD = 7.5      # Desired effective thermal conductivity, [W/m/K]
TOL_k = 0.5       # Bulk modulus tolerance, []
TOL_mu = 0.5      # Shear modulus tolerance, []
TOL_K = 0.5       # Thermal tolerance, []
TOL_sig = 0.8     # Electrical tolerance, []

## Given Search Bounds for Phase 2 Material
k2min = k1           # Lower bound for Phase 2 bulk modulus, [Pa]
k2max = 10*k1        # Upper bound for bulk modulus, [Pa]
mu2min = mu1         # Lower bound for shear modulus, [Pa]
mu2max = 10*mu1      # Upper bound for shear modulus, [Pa]
K2min = K1           # Lower bound for thermal conductivity, [W/m/K]
K2max = 10*K1        # Upper bound for thermal conductivity, [W/m/K]
sig2Emin = sig1E     # Lower bound for electrical conductivity, [S/m]
sig2Emax = 10*sig1E  # Upper bound for electrical conductivity, [S/m]
v2min = 0            # Lower bound for volume fraction, [fraction]
v2max = 2/3            # Upper bound for volume fraction, [fraction]

## Given Cost Function Weights
W1 = 1/3         # Electrical weight, []
W2 = 1/3          # Thermal weight, []
W3 = 1/3          # Mechanical weight, []
w1 = 1          # Material property cost function weights, []
wj = 0.5          # Concentration tensor weights, []
gamma = 0.5       # Hashin-Shtrikman averaging constant, []

In [ ]:
## Genetic Algorithm Helper Functions

def sort(pi):
    new_pi = np.sort(pi, axis=0)
    ind = np.argsort(pi, axis=0)
    return [new_pi, ind]

def reorder(Lambda, ind):
    temp = np.zeros((S,dv))
    for i in range(0, len(ind)):
        temp[i,:] = Lambda[int(ind[i]),:]
    Lambda = temp
    return Lambda

def get_unique_designs(Lambda, new_cost):
    
    # Obtain Unique Strings and Costs
    [unique_costs, iorig, iuniq] = np.unique(new_cost)
    unique_mask = iuniq > 1
    unique_costs = new_cost[:, unique_mask]
    unique_strings = Lambda[:, unique_mask]

    # Mask excludes the first unique value, correct here
    unique_strings = [Lambda[:,1], unique_strings]
    unique_costs = [new_cost[:,1], unique_costs]

    return [unique_strings, unique_costs] 


# Steps 1-6

In [ ]:
## Cost Function Definition
# An anonymous function for evaluating fitness
def Pi(Lambda):

    ######## Step 1 ################
    # Extract design parameters from genetic string
    k2 = Lambda[0]    # Phase 2 bulk modulus, [Pa]
    mu2 = Lambda[1]   # Phase 2 shear modulus, [Pa]
    sig2E = Lambda[2] # Phase 2 electrical conductivity, [S/m]
    K2 = Lambda[3]   # Phase 2 thermal conductivity, [W/m/K]
    v2 = Lambda[4]    # Phase 2 volume fraction, []
    v1 = 1 - Lambda[4]    # Phase 1 volume fraction, []
    ################################    

    ######## Step 2 ################
    ## Min and Max Effective Property Bounds
    # Bounds on electrical conductivity
    sigE_min = sig1E + v2 / (1/(sig2E-sig1E) + (1-v2)/(3*sig1E))
    sigE_max = sig2E + (1-v2) / (1/(sig1E-sig2E) + v2/(3*sig2E))
    
    # Bounds on thermal conductivity
    K_min = K1 + v2 / (1/(K2-K1) + (1-v2)/(3*K1)) 
    K_max = K2 + (1-v2) / (1/(K1-K2) + v2/(3*K2))
    
    # Bounds on bulk modulus
    k_min = k1 + v2 / (1/(k2-k1) + 3*(1-v2)/(3*k1 + 4*mu1))
    k_max = k2 + (1-v2) / (1/(k1-k2) + 3*v2/(3*k2 + 4*mu2)) 
    
    # Bounds on shear modulus
    mu_min = mu1 + v2 / (1/(mu2-mu1) + 6*(1-v2)*(k1+2*mu1)/(5*mu1*(3*k1+4*mu1)))
    mu_max = mu2 + (1-v2) / (1/(mu1-mu2) + 6*v2*(k2+2*mu2)/(5*mu2*(3*k2+4*mu2)))
    ################################

    ######## Step 3 ################
    ## Effective Material Properties
    sigE_eff = gamma * (sigE_min + sigE_max)
    K_eff    = gamma * (K_min + K_max)
    k_eff    = gamma * (k_min + k_max)
    mu_eff   = gamma * (mu_min + mu_max)
    ################################

    ######## Step 4 ################
    ## Concentration Tensor Determination
    # Electrical Load Sharing
    CJ1CE1 = (sig1E / sigE_eff) * ((1 / (1-v2)) * (sig2E-sigE_eff) / (sig2E-sig1E))**2
    CJ2CE2 = (sig2E / sigE_eff) * ((1 / v2) * (sigE_eff-sig1E) / (sig2E-sig1E))**2
    
    # Thermal Load Sharing
    Ct2 = (1 / v2) * (K2 - K1)**-1 * (K_eff - K1)
    Ct1 = (1 / v1) * (1 - v2*Ct2)
    Cq2 = K2 * Ct2 * K_eff**-1
    Cq1 = (1 - v2 * Cq2) / (1 - v2)
        
    # Mechanical Load Sharing
    Ck2  = (1 / v2) * (k2 / k_eff) * ((k_eff - k1) / (k2 - k1))
    Cmu2 = (1 / v2) * (mu2 / mu_eff) * ((mu_eff - mu1) / (mu2 - mu1))
    Ck1  = (1 / v1) * (1 - v2 * Ck2)
    Cmu1 = (1 / v1) * (1 - v2 * Cmu2)
    ################################

    ## Cost Function Weight Determination
    # Assign concentration tensor weights
    w2E = wj
    w3E = wj
    w2T = wj
    w3T = wj
    w3M = wj
    w4M = wj
    w5M = wj
    w6M = wj
    
    ######## Step 5 ################
    # Electrical weights
    if :
        w2E_hat = 
    else:
        w2E_hat = 0

    if :
        w3E_hat = 
    else:
        w3E_hat = 0
    
    # Thermal weights
    if :
        w2T_hat = 
    else:
        w2T_hat = 0

    if :
        w3T_hat = 
    else:
        w3T_hat = 0
    
    # Mechanical weights
    if :
        w3M_hat = 
    else:
        w3M_hat = 0

    if :
        w4M_hat = 
    else:
        w4M_hat = 0

    if :
        w5M_hat = 
    else:
        w5M_hat = 0

    if :
        w6M_hat = 
    else:
        w6M_hat = 0
    ################################

    ######## Step 6 ################
    ## Cost Functions
    # Electrical Property Cost Function
    Pi_elec = 

    # Thermal Property Cost Function    
    Pi_thermo = 
        
    # Mechanical Property Cost Function
    Pi_mech = 

    # Multi-property (Combined) Cost Function
    cost = 

    return cost
    ################################

# Question 1 Case A

In [ ]:
# - Keep the top P=10 parents after each generation
# - Produce 10 offspring
# - Generate 180 new random strings

def caseA():
    
    # Initialize arrays to store the cost and original indices of each generation
    PI = np.ones((G, S))
    Orig = np.ones((G, S))
    
    # Initialize arrays to store best performer and parent avg 
    Pi_min = np.zeros(G)     # best cost
    Pi_par_avg = np.zeros(G) # avg cost of parents
    
    # Generation counter
    g = 0

    # Initialize array to store costs for current generation
    cost = np.zeros(S)

    # Randomly populate first generation  
    Lambda = np.ones((S, dv))
    for i in range (S):
        k2    = np.random.uniform(k2min, k2max)
        mu2   = np.random.uniform(mu2min, mu2max)
        sig2E = np.random.uniform(sig2Emin, sig2Emax)
        K2    = np.random.uniform(K2min, K2max)
        v2    = np.random.uniform(v2min, v2max)  
            
        # Evaluate initial population
        Lambda[i, :] = k2, mu2, sig2E, K2, v2
        cost[i] = Pi(Lambda[i, :])
            
    # Sort the costs of the first generation
    [new_cost, ind] = sort(cost)  
    PI[g, :] = new_cost.reshape(1,S) 
    
    # Store the cost of the best performer and average cost of the parents 
    Pi_min[g] = np.min(new_cost)
    Pi_par_avg[g] = np.mean(new_cost[0:P])
    
    # Update Lambda based on sorted indices
    Lambda = reorder(Lambda, ind)
    Orig[g,:] = ind.reshape(1,S) 
    
    # Perform all later generations    
    while g < G:
        
        print(g)
        cost[0:P] = new_cost[0:P] # retain the parents from the previous generation
        
        # Select top parents P from Lambda to be breeders
        for p in range(0, P, 2):
            phi1, phi2 = np.random.rand(2)
            kid1 = phi1*Lambda[p, :] + (1-phi1)*Lambda[p+1, :]
            kid2 = phi2*Lambda[p, :] + (1-phi2)*Lambda[p+1, :]
            
            # Append offspring to Lambda 
            Lambda[P+p,   :] = kid1
            Lambda[P+p+1, :] = kid2
        
            # Evaluate Offspring
            cost[P+p]   = Pi(Lambda[P+p, :])
            cost[P+p+1] = Pi(Lambda[P+p+1, :])
                     
        # Randomly generate new design strings to fill the rest of the population
        for i in range(S-P-K):
            k2    = np.random.uniform(k2min, k2max)
            mu2   = np.random.uniform(mu2min, mu2max)
            sig2E = np.random.uniform(sig2Emin, sig2Emax)
            K2    = np.random.uniform(K2min, K2max)
            v2    = np.random.uniform(v2min, v2max)  
            
            # Evaluate population
            Lambda[P+K+i, :] = k2, mu2, sig2E, K2, v2
            cost[P+K+i] = Pi(Lambda[P+K+i, :])

        # Sort the costs for the gth generation
        [new_cost, ind] = sort(cost)  
        PI[g, :] = new_cost.reshape(1,S) 
    
        # Store the cost of the best performer and average cost of the parents 
        Pi_min[g] = np.min(new_cost)
        Pi_par_avg[g] = np.mean(new_cost[0:P])
    
        # Update Lambda based on sorted indices
        Lambda = reorder(Lambda, ind)
        Orig[g,:] = ind.reshape(1,S)  

        # Update the generation counter
        g = g + 1          
    
    return PI, Lambda, Orig, Pi_min, Pi_par_avg   


In [ ]:
# Call the Genetic Algorithm for Case A  
PI, Lambda, Orig, Pi_min, Pi_par_avg = caseA() # Keep the top P=10 parents after each generation
                                               # Produce 10 offspring
                                               # Generate 180 new random strings

caseA_results = {
    "PI": PI,
    "Lambda": Lambda,
    "Orig": Orig,
    "Pi_min": Pi_min,
    "Pi_par_avg": Pi_par_avg,
}

with open("caseA_results.pkl", "wb") as f:
    pickle.dump(caseA_results, f)

In [ ]:
# Plotting
if 'caseA_results' not in globals():
    with open("caseA_results.pkl", "rb") as f:
        caseA_results = pickle.load(f)

PI = caseA_results["PI"]
Lambda = caseA_results["Lambda"]
Orig = caseA_results["Orig"]
Pi_min = caseA_results["Pi_min"]
Pi_par_avg = caseA_results["Pi_par_avg"]
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(range(G), Pi_par_avg, label='top 10 avg')
ax.plot(range(G), Pi_min, label="best ")
plt.xlabel('Generation', fontsize= 20)
plt.ylabel('Cost', fontsize=20)
plt.title('Genetic Algorithm Results, Case A', fontsize = 24)
plt.legend(fontsize = 14)
plt.show()

# Question 1 Case B

In [ ]:
# - Do NOT keep top P=10 parents after each generation
# - Produce 10 offspring
# - Generate 190 new random strings

def caseB():
    
    # Initialize arrays to store the cost and original indices of each generation
    PI = np.ones((G, S))
    Orig = np.ones((G, S))
    
    # Initialize arrays to store best performer and parent avg 
    Pi_min = np.zeros(G)     # best cost
    Pi_par_avg = np.zeros(G) # avg cost of parents
    
    # Generation counter
    g = 0

    # Initialize array to store costs for current generation
    cost = np.zeros(S)

    # Randomly populate first generation  
    Lambda = np.ones((S, dv))
    for i in range (S):
        k2    = np.random.uniform(k2min, k2max)
        mu2   = np.random.uniform(mu2min, mu2max)
        sig2E = np.random.uniform(sig2Emin, sig2Emax)
        K2    = np.random.uniform(K2min, K2max)
        v2    = np.random.uniform(v2min, v2max)  
            
        # Evaluate initial population
        Lambda[i, :] = k2, mu2, sig2E, K2, v2
        cost[i] = Pi(Lambda[i, :])
            
    # Sort the costs of the first generation
    [new_cost, ind] = sort(cost)  
    PI[g, :] = new_cost.reshape(1,S) 
    
    # Store the cost of the best performer and average cost of the parents 
    Pi_min[g] = np.min(new_cost)
    Pi_par_avg[g] = np.mean(new_cost[0:P])
    
    # Update Lambda based on sorted indices
    Lambda = reorder(Lambda, ind)
    Orig[g,:] = ind.reshape(1,S) 
    g += 1
    
    # Perform all later generations    
    while g < G:
        
        print(g)

        ######## Step 7 ################
        cost =  # do NOT retain the parents from the previous generation
        
        # Select top parents P from Lambda to be breeders
        for p in range(0, P, 2):
            phi1, phi2 = np.random.rand(2)
            kid1 = phi1*Lambda[p, :] + (1-phi1)*Lambda[p+1, :]
            kid2 = phi2*Lambda[p, :] + (1-phi2)*Lambda[p+1, :]
            
            # Append offspring to Lambda (children fill in spots where parents would be)
            Lambda[ ,   :] = kid1
            Lambda[ ,   :] = kid2
        
            # Evaluate Offspring
            cost[ ] = Pi(Lambda[p, :])
            cost[ ] = Pi(Lambda[p+1, :])
        ################################
              
        # Randomly generate new design strings to fill th rest of the population
        for i in range(S-K):
            k2    = np.random.uniform(k2min, k2max)
            mu2   = np.random.uniform(mu2min, mu2max)
            sig2E = np.random.uniform(sig2Emin, sig2Emax)
            K2    = np.random.uniform(K2min, K2max)
            v2    = np.random.uniform(v2min, v2max)  
            
            # Evaluate population
            Lambda[K+i, :] = k2, mu2, sig2E, K2, v2
            cost[K+i] = Pi(Lambda[K+i, :])

        # Sort the costs for the gth generation
        [new_cost, ind] = sort(cost)  
        PI[g, :] = new_cost.reshape(1,S) 
    
        # Store the cost of the best performer and average cost of the parents 
        Pi_min[g] = np.min(new_cost)
        Pi_par_avg[g] = np.mean(new_cost[0:P])
    
        # Update Lambda based on sorted indices
        Lambda = reorder(Lambda, ind)
        Orig[g,:] = ind.reshape(1,S)  

        # Update the generation counter
        g = g + 1          
    
    return PI, Lambda, Orig, Pi_min, Pi_par_avg   


In [ ]:
# Call the Genetic Algorithm for Case B  
PI, Lambda, Orig, Pi_min, Pi_par_avg = caseB() # Do NOT keep the top P=10 parents after each generation
                                               # Produce 10 offspring
                                               # Generate 190 new random strings

caseB_results = {
    "PI": PI,
    "Lambda": Lambda,
    "Orig": Orig,
    "Pi_min": Pi_min,
    "Pi_par_avg": Pi_par_avg,
}

with open("caseB_results.pkl", "wb") as f:
    pickle.dump(caseB_results, f)

In [ ]:
# Plotting
if 'caseB_results' not in globals():
    with open("caseB_results.pkl", "rb") as f:
        caseB_results = pickle.load(f)
        
fig, ax = plt.subplots(figsize=(10,6))
ax.plot(range(G), Pi_par_avg, label='top 10 avg')
ax.plot(range(G), Pi_min, label="best ")
plt.xlabel('Generation', fontsize= 20)
plt.ylabel('Cost', fontsize=20)
plt.title('Genetic Algorithm Results, Case B', fontsize = 24)
plt.legend(fontsize = 14)
plt.show()

# Question 1 Case C

In [ ]:
# - Keep top P=10 parents after each generation
# - Produce 10 offspring with mutations using random weights [-1/2, 3/2] instead of [0, 1]
# - Generate 180 new random strings

######## Step 8 ################
a =
b = 
################################

def caseC():
    
    # Initialize arrays to store the cost and original indices of each generation
    PI = np.ones((G, S))
    Orig = np.ones((G, S))
    
    # Initialize arrays to store best performer and parent avg 
    Pi_min = np.zeros(G)     # best cost
    Pi_par_avg = np.zeros(G) # avg cost of parents
    
    # Generation counter
    g = 0

    # Initialize array to store costs for current generation
    cost = np.zeros(S)

    # Randomly populate first generation  
    Lambda = np.ones((S, dv))
    for i in range (S):
        k2    = np.random.uniform(k2min, k2max)
        mu2   = np.random.uniform(mu2min, mu2max)
        sig2E = np.random.uniform(sig2Emin, sig2Emax)
        K2    = np.random.uniform(K2min, K2max)
        v2    = np.random.uniform(v2min, v2max)  
            
        # Evaluate initial population
        Lambda[i, :] = k2, mu2, sig2E, K2, v2
        cost[i] = Pi(Lambda[i, :])
            
    # Sort the costs of the first generation
    [new_cost, ind] = sort(cost)  
    PI[g, :] = new_cost.reshape(1,S) 
    
    # Store the cost of the best performer and average cost of the parents 
    Pi_min[g] = np.min(new_cost)
    Pi_par_avg[g] = np.mean(new_cost[0:P])
    
    # Update Lambda based on sorted indices
    Lambda = reorder(Lambda, ind)
    Orig[g,:] = ind.reshape(1,S) 
    
    g += 1
    # Perform all later generations    
    while g < G:
        
        print(g)
        cost[0:P] = new_cost[0:P] # retain the parents from the previous generation
        
        ######## Step 9 ################
        # Select top parents P from Lambda to be breeders
        for p in range(0, P, 2):
            phi1 = 
            phi2 = 
            kid1 = phi1*Lambda[p, :] + (1-phi1)*Lambda[p+1, :]
            kid2 = phi2*Lambda[p, :] + (1-phi2)*Lambda[p+1, :]
            
            # Append offspring to Lambda 
            Lambda[ ,   :] = kid1
            Lambda[ , :] = kid2
        
            # Evaluate Offspring
            cost[ ]   = Pi(Lambda[P+p, :])
            cost[ ] = Pi(Lambda[P+p+1, :])
        ################################
                 
        # Randomly generate new design strings to fill th rest of the population
        for i in range(S-P-K):
            k2    = np.random.uniform(k2min, k2max)
            mu2   = np.random.uniform(mu2min, mu2max)
            sig2E = np.random.uniform(sig2Emin, sig2Emax)
            K2    = np.random.uniform(K2min, K2max)
            v2    = np.random.uniform(v2min, v2max)  
            
            # Evaluate population
            Lambda[P+K+i, :] = k2, mu2, sig2E, K2, v2
            cost[P+K+i] = Pi(Lambda[P+K+i, :])

        # Sort the costs for the gth generation
        [new_cost, ind] = sort(cost)  
        PI[g, :] = new_cost.reshape(1,S) 
    
        # Store the cost of the best performer and average cost of the parents 
        Pi_min[g] = np.min(new_cost)
        Pi_par_avg[g] = np.mean(new_cost[0:P])
    
        # Update Lambda based on sorted indices
        Lambda = reorder(Lambda, ind)
        Orig[g,:] = ind.reshape(1,S)  

        # Update the generation counter
        g += 1          
    
    return PI, Lambda, Orig, Pi_min, Pi_par_avg   


In [ ]:
# Call the Genetic Algorithm for Case C  
PI, Lambda, Orig, Pi_min, Pi_par_avg = caseC() # Keep top P=10 parents after each generation
                                               # Produce 10 offspring with mutations using random weights [-1/2, 3/2] instead of [0, 1]
                                               # Generate 180 new random strings 
                                               

caseC_results = {
    "PI": PI,
    "Lambda": Lambda,
    "Orig": Orig,
    "Pi_min": Pi_min,
    "Pi_par_avg": Pi_par_avg,
}

with open("caseC_results.pkl", "wb") as f:
    pickle.dump(caseC_results, f)

In [ ]:
# Plotting
if 'caseC_results' not in globals():
    with open("caseC_results.pkl", "rb") as f:
        caseC_results = pickle.load(f)

fig, ax = plt.subplots(figsize=(10,6))
ax.plot(range(G), Pi_par_avg, label='top 10 avg')
ax.plot(range(G), Pi_min, label="best ")
plt.xlabel('Generation', fontsize= 20)
plt.ylabel('Cost', fontsize=20)
plt.title('Genetic Algorithm Results, Case C', fontsize = 24)
plt.legend(fontsize = 14)
plt.show()

# Question 3

In [ ]:
if 'caseA_results' not in globals():
    with open("caseA_results.pkl", "rb") as f:
        caseA_results = pickle.load(f)
if 'caseB_results' not in globals():
    with open("caseB_results.pkl", "rb") as f:
        caseB_results = pickle.load(f)
if 'caseC_results' not in globals():
    with open("caseC_results.pkl", "rb") as f:
        caseC_results = pickle.load(f)

decimals = 4
print("Top 4 Designs Case A:")
for i in range(4):
    print(f'Lambda[{i}]')
    print(caseA_results['Lambda'][i,:])
    print(f'PI[{i}]')
    print(np.round(caseA_results['PI'][-1,i], decimals))
    print('\n ')

print('\n \n ')

print("Top 4 Designs Case B:")
for i in range(4):
    print(f'Lambda[{i}]')
    print(caseB_results['Lambda'][i,:])
    print(f'PI[{i}]')
    print(np.round(caseB_results['PI'][-1,i], decimals))
    print('\n ')

print('\n \n ')

print("Top 4 Designs Case C:")
for i in range(4):
    print(f'Lambda[{i}]')
    print(caseC_results['Lambda'][i,:])
    print(f'PI[{i}]')
    print(np.round(caseC_results['PI'][-1,i], decimals))
    print('\n ')